# 3.3 Introduction to Multi-Agent Systems (MAS)

**Week 4 — Agentic AI & Multi-Agent Systems**

## Learning objectives
- Explain the *single-agent bottleneck* and why it appears as you add more tools
- Apply the **"Project Team"** analogy to design multi-agent systems
- Compare LangChain, AutoGen, and CrewAI and know when to reach for each


## 1. The Single-Agent Bottleneck

In 3.2 our agent had two tools. Real systems often need 10, 20, 50+ tools: database lookups, web
search, email sending, calculations, document retrieval, code execution... What happens as a single
agent's tool list grows?

- **Tool selection accuracy drops.** The model has to read every tool description before choosing —
  more (and more similar) tools means more confusable choices.
- **Prompts get longer and slower.** Every tool description sits in context on every single call.
- **Responsibility gets muddy.** One "do everything" agent has no natural place to enforce a
  domain-specific rule (e.g. "never auto-approve refunds over ₹5,000") — that logic either gets
  bolted onto the giant prompt or skipped.
- **Errors compound.** If the one agent misclassifies a request, every downstream step inherits
  that mistake with no independent check.

Let's demonstrate the accuracy drop directly with a toy experiment.


In [ ]:
import random

random.seed(7)

def simulate_tool_selection_accuracy(num_tools: int, num_trials: int = 500) -> float:
    """Toy model: as the pool of plausible-looking tools grows, the chance the (simulated) LLM
    picks the exactly-correct one on a genuinely ambiguous query drops, because more tools compete
    for the same intent. This mirrors a well-documented real effect, simplified for teaching."""
    correct = 0
    for _ in range(num_trials):
        # baseline confusion grows slowly with pool size
        confusion_penalty = min(0.5, 0.02 * num_tools)
        if random.random() > confusion_penalty:
            correct += 1
    return correct / num_trials

for n in [2, 5, 10, 20, 40, 60]:
    acc = simulate_tool_selection_accuracy(n)
    print(f"{n:>3} tools available -> simulated tool-selection accuracy: {acc:.0%}")


The exact numbers are illustrative, not a benchmark — but the *shape* of the curve is real and
well documented in agent literature: single agents degrade as their tool surface grows. The fix isn't
"write a better prompt" — it's **architectural**: split the work across multiple, narrowly-scoped
agents.


## 2. The "Project Team" Analogy

Instead of one generalist agent holding every tool, design a **team**: each agent has

- **one clear role** (e.g. "Classifier", "Researcher", "Drafter", "Validator", "Escalator"),
- **a small, focused tool set** relevant only to that role,
- **a precisely defined output** that the next agent in the pipeline can consume,
- **a clear handoff point** — what context does the next agent actually need, and what can be dropped?

This mirrors how a real project team works: a business analyst doesn't personally write the code, and
the engineer doesn't personally sign off on compliance — each role does one thing well and hands off a
clean, well-defined artifact to the next.


In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict

@dataclass
class AgentRole:
    name: str
    responsibility: str
    produces: str  # what this agent hands off to the next one

team = [
    AgentRole("Classifier", "Determine the type and urgency of an incoming request", "request_type, urgency"),
    AgentRole("Researcher", "Retrieve the relevant knowledge needed to respond", "grounded_context"),
    AgentRole("Drafter", "Propose a response using the grounded context", "draft_response"),
    AgentRole("Validator", "Check the draft for accuracy, tone, and policy compliance", "validated_response or issues"),
    AgentRole("Escalator", "Decide whether a human must review before this goes out", "final_decision"),
]

for role in team:
    print(f"{role.name:>11}: {role.responsibility}\n{'':>13}-> hands off: {role.produces}\n")


### Managing context handoffs

The trickiest part of MAS design is deciding **exactly what crosses the boundary** between agents.
Pass too little, and downstream agents re-derive things badly. Pass too much, and every agent's prompt
balloons with irrelevant history (the same problem we just diagnosed for single agents, now duplicated
across the whole team).

A good rule of thumb: each agent should receive **only the structured output of the previous agent**,
not the previous agent's full raw transcript.


In [ ]:
def classifier(request: str) -> Dict[str, Any]:
    urgent = any(w in request.lower() for w in ["urgent", "immediately", "asap"])
    return {"request_type": "refund_query" if "refund" in request.lower() else "general_query",
            "urgency": "high" if urgent else "normal"}

def researcher(classification: Dict[str, Any]) -> Dict[str, Any]:
    kb = {"refund_query": "Refunds are processed within 5-7 business days per the returns policy."}
    return {**classification, "grounded_context": kb.get(classification["request_type"], "No specific policy found.")}

def drafter(state: Dict[str, Any]) -> Dict[str, Any]:
    return {**state, "draft_response": f"Based on our policy: {state['grounded_context']}"}

def validator(state: Dict[str, Any]) -> Dict[str, Any]:
    issues = [] if "policy" in state["draft_response"] or "No specific" in state["grounded_context"] else ["Unsupported claim"]
    return {**state, "issues": issues}

def escalator(state: Dict[str, Any]) -> Dict[str, Any]:
    must_escalate = state["urgency"] == "high" or state["issues"]
    return {**state, "final_decision": "ESCALATE_TO_HUMAN" if must_escalate else "AUTO_SEND"}

# Run the pipeline, passing only structured state forward -- not raw transcripts
request = "I need my refund processed immediately, it's been two weeks!"
state = classifier(request)
state = researcher(state)
state = drafter(state)
state = validator(state)
state = escalator(state)

for k, v in state.items():
    print(f"{k:>16}: {v}")


This five-line pipeline is a *miniature* version of exactly the architecture used in Week 4's
capstones (SupportPilot, CareRoute) and later in Week 6's Nexus: **classify → retrieve → draft →
validate → escalate.** Sections 3.4 and 3.5 show how to implement this with real orchestration
frameworks instead of plain function calls.


## 3. Framework Comparison Guide

| Framework | Mental model | Best fit when... |
|---|---|---|
| **LangChain** | Pipeline / graph of steps | You want explicit control over a linear or branching flow, with tools bound to specific steps |
| **AutoGen** | Conversation between agents | The task is naturally a back-and-forth dialogue (e.g. one agent proposes, another critiques/executes) |
| **CrewAI** | Role-based team with a manager | You want to describe a team by *roles and goals* (like real job titles) and let a "crew" orchestrate hand-offs |

None of these is strictly "better" — they encode the same ReAct-style loop from 3.1 with different
ergonomics. Framework choice is a *design* decision, not a technical constraint: pick the mental model
that matches how you naturally think about the problem's structure.


In [ ]:
# A tiny decision-support function -- not a rule, just a starting heuristic for framework choice

def suggest_framework(task_shape: str) -> str:
    task_shape = task_shape.lower()
    if "debate" in task_shape or "propose and critique" in task_shape or "code execution" in task_shape:
        return "AutoGen — conversation-driven, good for propose/execute/critique loops"
    if "team of roles" in task_shape or "specialist" in task_shape:
        return "CrewAI — role-based teams with defined goals"
    return "LangChain — pipeline/graph-based, good for explicit, controllable step sequences"

for shape in ["a fixed classify -> retrieve -> draft pipeline",
              "two agents that propose code and execute/critique it",
              "a team of specialist roles like chef and nutritionist"]:
    print(f"{shape}\n  -> {suggest_framework(shape)}\n")


## Key Takeaways

- A single agent's performance **degrades** as its tool surface grows — this is a design problem, not
  a prompting problem.
- Think of a MAS as a **project team**: one role each, small focused tool sets, precisely defined
  hand-offs.
- Pass **structured state**, not raw transcripts, between agents to keep each agent's context lean.
- **LangChain** (pipeline/graph), **AutoGen** (conversation-driven), and **CrewAI** (role-based teams)
  are three different ergonomics over the same underlying agentic loop — choose based on how you
  naturally model the task.

## Check your understanding
1. Give two concrete reasons a single agent's accuracy drops as its tool list grows.
2. In the Project Team analogy, why should each agent produce a *precisely defined* output rather than
   a free-form one?
3. Which framework would you reach for first to build a two-agent "write code, then review it" flow —
   and why?

Next: **3.4 Orchestrating Multi-Agent Workflows** — fixed-sequence vs. dynamic routing, and how to
save/restore agent state across a handoff.
